In [22]:
import pandas as pd
import numpy as np
import re
import math
from collections import Counter

In [23]:
text = """If I drive for you, you give me a time and a place, I give you a five-minute window.

Anything happens in that five minutes and I’m yours. No matter what.

Anything happens a minute either side of that and you’re on your own. Do you understand?"""


In [24]:
#clean the data
documents = [p.strip() for p in text.split('\n\n') if p.strip()]

stop_words = set([
    'the', 'is', 'at', 'which', 'on', 'for', 'to', 'be', 'with', 'a', 'an', 'and', 'of', 'this', 'from', 'i', 'you',
    'in', 'it', 'that', 'just', 'your', 'have', 'has', 'as', 'are', 'was', 'were', 'but', 'by', 'not', 'or', 'so', 'if'
])

def tokenize(text):
    tokens = re.findall(r'\b\w+\b', text.lower())
    tokens = re.findall(r'\b\w+\b', text.lower())
    return [t for t in tokens if t not in stop_words and len(t)>2]

tokenized_docs = [tokenize(doc) for doc in documents]
all_terms = sorted(set(term for doc in tokenized_docs for term in doc))



In [30]:
documents

['If I drive for you, you give me a time and a place, I give you a five-minute window.',
 'Anything happens in that five minutes and I’m yours. No matter what.',
 'Anything happens a minute either side of that and you’re on your own. Do you understand?']

In [25]:
all_terms

['anything',
 'drive',
 'either',
 'five',
 'give',
 'happens',
 'matter',
 'minute',
 'minutes',
 'own',
 'place',
 'side',
 'time',
 'understand',
 'what',
 'window',
 'yours']

In [26]:
# TF calculation

def compute_idf(tokenized_documents):
    N = len(tokenized_documents)
    all_terms = set(term for doc in tokenized_documents for term in doc)
    idf = {}
    for term in all_terms:
        containing_docs = sum(1 for doc in tokenized_documents if term in doc)
        idf[term] = math.log((N + 1) / (containing_docs + 1)) + 1
    return idf

idf = compute_idf(tokenized_docs)



In [27]:
idf

{'minute': 1.2876820724517808,
 'happens': 1.2876820724517808,
 'yours': 1.6931471805599454,
 'window': 1.6931471805599454,
 'side': 1.6931471805599454,
 'give': 1.6931471805599454,
 'matter': 1.6931471805599454,
 'place': 1.6931471805599454,
 'own': 1.6931471805599454,
 'what': 1.6931471805599454,
 'either': 1.6931471805599454,
 'five': 1.2876820724517808,
 'drive': 1.6931471805599454,
 'understand': 1.6931471805599454,
 'minutes': 1.6931471805599454,
 'time': 1.6931471805599454,
 'anything': 1.2876820724517808}

In [28]:
def compute_tfidf(tf, idf):
    return {term: tf[term] * idf[term] for term in all_terms}

tfidf_docs = [compute_tfidf(tf_doc, idf) for tf_doc in tf_docs]



In [ ]:
tfidf_docs[0]

{'anything': 0.18395458177882582,
 'drive': 0.0,
 'either': 0.24187816865142076,
 'five': 0.0,
 'give': 0.0,
 'happens': 0.18395458177882582,
 'matter': 0.0,
 'minute': 0.18395458177882582,
 'minutes': 0.0,
 'own': 0.24187816865142076,
 'place': 0.0,
 'side': 0.24187816865142076,
 'time': 0.0,
 'understand': 0.24187816865142076,
 'what': 0.0,
 'window': 0.0,
 'yours': 0.0}

In [33]:
tfidf_docs[1]


{'anything': 0.18395458177882582,
 'drive': 0.0,
 'either': 0.0,
 'five': 0.18395458177882582,
 'give': 0.0,
 'happens': 0.18395458177882582,
 'matter': 0.24187816865142076,
 'minute': 0.0,
 'minutes': 0.24187816865142076,
 'own': 0.0,
 'place': 0.0,
 'side': 0.0,
 'time': 0.0,
 'understand': 0.0,
 'what': 0.24187816865142076,
 'window': 0.0,
 'yours': 0.24187816865142076}

In [35]:
#tfidf matrix for cosine similarity
tfidf_matrix = np.array([[doc[term] for term in all_terms] for doc in tfidf_docs])

In [36]:
tfidf_matrix

array([[0.        , 0.2116434 , 0.        , 0.16096026, 0.4232868 ,
        0.        , 0.        , 0.16096026, 0.        , 0.        ,
        0.2116434 , 0.        , 0.2116434 , 0.        , 0.        ,
        0.2116434 , 0.        ],
       [0.18395458, 0.        , 0.        , 0.18395458, 0.        ,
        0.18395458, 0.24187817, 0.        , 0.24187817, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.24187817,
        0.        , 0.24187817],
       [0.18395458, 0.        , 0.24187817, 0.        , 0.        ,
        0.18395458, 0.        , 0.18395458, 0.        , 0.24187817,
        0.        , 0.24187817, 0.        , 0.24187817, 0.        ,
        0.        , 0.        ]])

In [37]:
#cosine similarity function

def cosine_similarity(vec1, vec2):
    num = np.dot(vec1,vec2)
    denom = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    return num / denom if denom != 0 else 0.0


In [38]:
for i in range(len(tfidf_matrix)):
    for j in range(len(tfidf_matrix)):
        print(f"Doc {i+1} vs Doc {j+1}: {cosine_similarity(tfidf_matrix[i], tfidf_matrix[j]):.4f}")
    

Doc 1 vs Doc 1: 1.0000
Doc 1 vs Doc 2: 0.0798
Doc 1 vs Doc 3: 0.0798
Doc 2 vs Doc 1: 0.0798
Doc 2 vs Doc 2: 1.0000
Doc 2 vs Doc 3: 0.2017
Doc 3 vs Doc 1: 0.0798
Doc 3 vs Doc 2: 0.2017
Doc 3 vs Doc 3: 1.0000


In [ ]:
#bool